In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import time
import os
from dataclasses import dataclass

# --- CRAB CONFIGURATION ---
@dataclass
class CRABConfig:
    vocab_size: int = 50257    # GPT-2 standard vocab
    block_size: int = 512      # Context window (Optimized for T4 VRAM)
    n_embd: int = 512          # Embedding dimension
    n_head: int = 8            # Multi-head attention count (64 dim per head)
    n_layer: int = 14          # Number of transformer blocks
    dropout: float = 0.1       # Regularization

    # Training Hyperparameters
    batch_size: int = 4        # Micro-batch size (Small to prevent OOM)
    grad_accum_steps: int = 16 # Effective batch size = 4 * 16 = 64
    learning_rate: float = 6e-4
    max_iters: int = 2000
    weight_decay: float = 0.1

config = CRABConfig()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Engine Status: Ready on {device.upper()}")

Engine Status: Ready on CUDA


In [2]:
from transformers import GPT2Tokenizer
from datasets import load_dataset

# We use the GPT-2 tokenizer purely as a sub-word mapping tool
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Stream the dataset to save Colab RAM
print("Loading TinyStories stream...")
ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
data_iter = iter(ds)

def get_batch():
    """Generates X (input) and Y (target) tensors for training."""
    xs, ys = [], []
    while len(xs) < config.batch_size:
        try:
            text = next(data_iter)['text']
            tokens = tokenizer.encode(text)
            if len(tokens) > config.block_size + 1:
                # Pick a random chunk
                start = torch.randint(0, len(tokens) - config.block_size - 1, (1,)).item()
                chunk = tokens[start:start + config.block_size + 1]
                xs.append(torch.tensor(chunk[:-1]))
                ys.append(torch.tensor(chunk[1:]))
        except StopIteration:
            break
    return torch.stack(xs).to(device), torch.stack(ys).to(device)

print("Data Pipeline: Active")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading TinyStories stream...


README.md: 0.00B [00:00, ?B/s]

Data Pipeline: Active


In [6]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        # FIX: We must store the config to access it in the forward pass
        self.config = config

        self.n_head = config.n_head
        self.n_embd = config.n_embd

        # Combined Q, K, V projection for efficiency
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)

        # Regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.size()

        # Calculate query, key, values for all heads in batch and move head forward to be the batch dim
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # Reshape for multi-head: (B, T, Heads, Head_Dim) -> (B, Heads, T, Head_Dim)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        # Efficient Flash Attention (built-in to PyTorch 2.0+)
        # This is where the error was triggered. Now self.config exists!
        y = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=True,
            dropout_p=self.config.dropout if self.training else 0
        )

        # Re-assemble all head outputs side by side
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # Output projection
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return self.dropout(x)

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        # Using Pre-Norm architecture for better gradient flow
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [7]:
class CRAB(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight Tying: Share weights between embedding and output head to save 25M params
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

model = CRAB(config).to(device)
params = sum(p.numel() for p in model.parameters())
print(f"CRAB Parameters: {params/1e6:.2f}M")

CRAB Parameters: 70.06M


In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
scaler = torch.amp.GradScaler('cuda')

print("Starting Training Loop...")
model.train()

for i in range(config.max_iters):
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)

    # Gradient Accumulation
    for _ in range(config.grad_accum_steps):
        x, y = get_batch()
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits, loss = model(x, y)
            loss = loss / config.grad_accum_steps
        scaler.scale(loss).backward()
        # Explicitly delete logits to free VRAM
        del logits

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if i % 10 == 0:
        t1 = time.time()
        print(f"Step {i} | Loss: {loss.item()*config.grad_accum_steps:.4f} | Time: {(t1-t0)*1000:.2f}ms")

# Save the final brain
torch.save(model.state_dict(), "crab_v1.pth")
print("Model Trained and Saved as crab_v1.pth!")

Starting Training Loop...


Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


Step 0 | Loss: 10.9151 | Time: 2653.81ms
Step 10 | Loss: 6.5406 | Time: 2904.90ms
Step 20 | Loss: 5.7244 | Time: 2480.08ms
Step 30 | Loss: 5.3366 | Time: 3313.62ms
Step 40 | Loss: 5.2142 | Time: 2529.50ms
Step 50 | Loss: 5.1495 | Time: 1614.24ms
Step 60 | Loss: 5.1502 | Time: 2042.98ms
Step 70 | Loss: 5.2487 | Time: 2163.01ms
Step 80 | Loss: 4.9519 | Time: 2082.00ms
Step 90 | Loss: 4.5837 | Time: 2103.36ms
Step 100 | Loss: 4.3607 | Time: 2688.45ms
Step 110 | Loss: 4.2735 | Time: 3021.22ms
Step 120 | Loss: 4.1459 | Time: 2988.15ms
Step 130 | Loss: 4.2763 | Time: 2331.51ms
Step 140 | Loss: 4.1545 | Time: 1994.53ms
Step 150 | Loss: 4.1317 | Time: 1753.48ms
Step 160 | Loss: 3.8963 | Time: 1976.70ms
Step 170 | Loss: 3.9093 | Time: 1774.17ms
Step 180 | Loss: 3.6365 | Time: 3682.77ms
Step 190 | Loss: 3.9382 | Time: 4401.45ms
Step 200 | Loss: 3.8334 | Time: 2431.21ms
Step 210 | Loss: 3.6830 | Time: 5138.80ms
Step 220 | Loss: 3.4040 | Time: 3025.84ms
Step 230 | Loss: 3.7898 | Time: 3468.43ms
St

RuntimeError: stack expects a non-empty TensorList

In [9]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    """
    Generates text from a prompt.
    temperature: > 1.0 = more random, < 1.0 = more predictable
    top_k: limits the vocabulary to the top K most likely next words
    """
    model.eval()
    # Convert prompt to tokens
    idx = tokenizer.encode(prompt, return_tensors="pt").to(device)

    for _ in range(max_new_tokens):
        # Crop idx to the last 'block_size' tokens
        idx_cond = idx[:, -config.block_size:]

        # Get predictions
        logits, _ = model(idx_cond)
        # Focus only on the last time step and scale by temperature
        logits = logits[:, -1, :] / temperature

        # Optional: Top-K filtering
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float('Inf')

        # Apply softmax to get probabilities
        probs = F.softmax(logits, dim=-1)

        # Sample from the distribution
        idx_next = torch.multinomial(probs, num_samples=1)

        # Append to the sequence
        idx = torch.cat((idx, idx_next), dim=1)

        # If we hit the End-of-Sequence token, stop early
        if idx_next.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(idx[0].tolist())

# --- TEST IT OUT ---
my_prompt = "Once upon a time, a small crab named Arshvir"
generated_text = generate(model, my_prompt, max_new_tokens=50)

print(f"\n🦀 CRAB SAYS:\n{'-'*20}\n{generated_text}\n{'-'*20}")


🦀 CRAB SAYS:
--------------------
Once upon a time, a small crab named Arshvir in the water. The worm made a loud noise and tried to pull the worm in its mouth.

"Uh-oh, Tim, you can't touch the worm!" Kim said, feeling scared. "You have to stay with the worm
--------------------


In [14]:
import json
import os
from google.colab import files

# 1. Save the configuration dictionary
config_dict = {
    "vocab_size": config.vocab_size,
    "block_size": config.block_size,
    "n_embd": config.n_embd,
    "n_head": config.n_head,
    "n_layer": config.n_layer,
    "dropout": config.dropout
}

with open("crab_config.json", "w") as f:
    json.dump(config_dict, f)

print("✅ Configuration saved as crab_config.json")

# 2. Ensure the weights are explicitly saved (in case Cell 5 didn't finish gracefully)
torch.save(model.state_dict(), "crab_v1.pth")
print("✅ Weights safely locked into crab_v1.pth")

# 3. Trigger browser download (Make sure your browser allows pop-up downloads from Colab)
try:
    print("Initiating download... Check your browser downloads folder.")
    files.download("crab_config.json")
    files.download("crab_v1.pth")
except Exception as e:
    print(f"⚠️ Colab automatic download failed. Please manually right-click the files in the left sidebar and click 'Download'. Error: {e}")

✅ Configuration saved as crab_config.json
✅ Weights safely locked into crab_v1.pth
Initiating download... Check your browser downloads folder.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# -----------------------------------------------------------------------------
# CRAB Metrics Evaluation Dashboard
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

def plot_training_metrics(loss_history, val_loss_history=None):
    """
    Generates a clean, Big Tech-style metrics dashboard.
    """
    plt.figure(figsize=(10, 5))
    plt.style.use('dark_background') # Fits the sleek, minimalist aesthetic

    # Plot Training Loss
    plt.plot(loss_history, label='Training Loss', color='#00d2ff', linewidth=2)

    if val_loss_history:
        plt.plot(val_loss_history, label='Validation Loss', color='#ff007f', linewidth=2, linestyle='--')

    plt.title('CRAB v1 Pre-training Loss Curve', fontsize=14, pad=15)
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Cross-Entropy Loss', fontsize=12)
    plt.grid(True, alpha=0.2)
    plt.legend()

    # Calculate Perplexity from the final loss
    final_loss = loss_history[-1]
    perplexity = np.exp(final_loss)

    plt.figtext(0.15, 0.15, f"Final Loss: {final_loss:.4f}\nEstimated Perplexity: {perplexity:.2f}",
                bbox=dict(facecolor='black', alpha=0.5), fontsize=12)

    plt.show()

# Note: If you saved your losses to a list during training (e.g., `train_losses`),
# simply call: plot_training_metrics(train_losses)

In [12]:
plt.show()

In [13]:
# -----------------------------------------------------------------------------
# CRAB Inference Testing Suite
# -----------------------------------------------------------------------------

test_prompts = [
    "Once upon a time, a small crab named Arshvir",
    "The big dog looked at the cat and said",
    "One day, a spaceship landed in the forest."
]

print("🦀 CRAB V1 INFERENCE TESTS 🦀\n" + "="*30)

# Test the model across different "creativity" levels
for prompt in test_prompts:
    print(f"\n[PROMPT]: {prompt}")

    # Low temperature (Predictable, safe, logical)
    safe_text = generate(model, prompt, max_new_tokens=40, temperature=0.5)
    print(f"[TEMP 0.5 - Safe]: {safe_text}")

    # High temperature (Creative, slightly chaotic)
    creative_text = generate(model, prompt, max_new_tokens=40, temperature=1.2)
    print(f"[TEMP 1.2 - Creative]: {creative_text}")
    print("-" * 30)

🦀 CRAB V1 INFERENCE TESTS 🦀

[PROMPT]: Once upon a time, a small crab named Arshvir
[TEMP 0.5 - Safe]: Once upon a time, a small crab named Arshvir.

"Wow, this is amazing!" Sam said. "Do you want to play with me?"

"Yes, I want to play with you," Sam said. "But I
[TEMP 1.2 - Creative]: Once upon a time, a small crab named Arshvir.

"Thank you, Mr. Lee," Mom said. She smiled and watched Sara.

"Hello, Lisa," Sara said. "You are very strong. Do you think they
------------------------------

[PROMPT]: The big dog looked at the cat and said
[TEMP 0.5 - Safe]: The big dog looked at the cat and said, "Hello, I am a king. I live in a big box. I am a king and I am a princess. I will make you feel happy."

The cat said, "
[TEMP 1.2 - Creative]: The big dog looked at the cat and said "Let to put the lion to the pig, but that does not eat. It only make a mess, or it might hurt."

Tim and his cat looked at the cow. Ben l
------------------------------

[PROMPT]: One day, a spaceship landed in the